# 1 - Black-Scholes Analytical Benchmark

Prices every option in the ASIANPAINT dataset with the closed-form Black-Scholes formula, giving the benchmark the neural networks must beat.

Calls use `C = S*N(d1) - X*exp(-rT)*N(d2)`; puts use the corresponding put formula. Time to expiry is `t/365`, and volatility comes from the dataset's `sigma` column.

The contract set is split by moneyness: `underlying_value > strike_price` for calls, `<` for puts.

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm

In [7]:
df= pd.read_excel('../data/ASIANPAINT_Dataset.xlsx')
df.info


<bound method DataFrame.info of             Date     Expiry   t  strike_price  underlying_value     sigma  \
0     2020-01-01 2020-01-30  29          1980            1793.2  0.008151   
1     2020-01-01 2020-01-30  29          1440            1793.2  0.008151   
2     2020-01-01 2020-01-30  29          2020            1793.2  0.008151   
3     2020-01-01 2020-01-30  29          1920            1793.2  0.008151   
4     2020-01-01 2020-01-30  29          1940            1793.2  0.008151   
...          ...        ...  ..           ...               ...       ...   
35586 2020-12-31 2021-02-25  56          2620            2764.5  0.015889   
35587 2020-12-31 2021-02-25  56          2640            2764.5  0.015889   
35588 2020-12-31 2021-02-25  56          2660            2764.5  0.015889   
35589 2020-12-31 2021-02-25  56          2680            2764.5  0.015889   
35590 2020-12-31 2021-02-25  56          2700            2764.5  0.015889   

            r   close  
0      0.0494    3.

In [8]:
df_call = df[df.underlying_value > df.strike_price]
df=df.drop(['Date','Expiry'],axis=1)
df_put=df[df.underlying_value < df.strike_price]
df.head()

,t,strike_price,underlying_value,sigma,r,close
0,29,1980,1793.2,0.008151,0.0494,3.8
1,29,1440,1793.2,0.008151,0.0494,398.5
2,29,2020,1793.2,0.008151,0.0494,1.2
3,29,1920,1793.2,0.008151,0.0494,6.5
4,29,1940,1793.2,0.008151,0.0494,5.0


In [13]:
def black_scholes(row):

    S = row.underlying_value
    X = row.strike_price
    T = row.t / 365
    r = row.r
    σ = row.sigma
    d1 = (np.log(S / X) + (r + (σ ** 2) / 2) * T) / (σ * (T ** .5))
    d2 = d1 - σ * (T ** .5)
    C = S * norm.cdf(d1) - X * np.exp(-r * T) * norm.cdf(d2)
    return C
def black_scholes_put(row):

    S = row.underlying_value
    X = row.strike_price
    T = row.t / 365
    r = row.r
    σ = row.sigma
    d1 = (np.log(S / X) + (r + (σ ** 2) / 2) * T) / (σ * (T ** .5))
    d2 = d1 - σ * (T ** .5)
    P  = norm.cdf(-d2) * X * np.exp(-r * T) - S * norm.cdf(-d1)
    return P


In [14]:
df_call['black_scholes_pred'] =df_call.apply(black_scholes, axis=1)
df_put['black_scholes_pred'] =df_put.apply(lambda row: black_scholes_put(row), axis=1)

<ipython-input-13-1651d1ee0908>:8: RuntimeWarning: divide by zero encountered in double_scalars
  d1 = (np.log(S / X) + (r + (σ ** 2) / 2) * T) / (σ * (T ** .5))
<ipython-input-14-b3a3914983c8>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_call['black_scholes_pred'] =df_call.apply(black_scholes, axis=1)
<ipython-input-13-1651d1ee0908>:19: RuntimeWarning: divide by zero encountered in double_scalars
  d1 = (np.log(S / X) + (r + (σ ** 2) / 2) * T) / (σ * (T ** .5))
<ipython-input-14-b3a3914983c8>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/inde

In [15]:


black_prid_call= df_call.black_scholes_pred
real_call=df_call.close
black_prid_put= df_put.black_scholes_pred
real_put=df_put.close


In [16]:
from sklearn.metrics import mean_squared_error,mean_absolute_error
accuracy = mean_squared_error(black_prid_call,real_call)
print(accuracy)
accuracy1=mean_absolute_error(black_prid_call,real_call)
print(accuracy1)
print(np.sqrt(accuracy))

34251.67726740593
127.8787502595569
185.07208667815343


In [17]:
accuracy = mean_squared_error(black_prid_put,real_put)
print(accuracy)
accuracy1=mean_absolute_error(black_prid_put,real_put)
print(accuracy1)
print(np.sqrt(accuracy))

38726.45411700674
160.00512410506488
196.79038115976792
